# CHSA Medical Triage Agent - Google Colab T4

This notebook is the Colab/T4 version of the training workflow. It avoids creating a `.venv`, uses Colab's built-in PyTorch/CUDA stack, downloads the private Hugging Face dataset, audits it, then runs full SFT and DPO adapter training with commented smoke commands kept for quick checks.

It can be run directly in Colab or from VSCode connected to a Colab Jupyter runtime. Use a T4 GPU runtime; uncomment a smoke command first when validating a fresh runtime.

## Runtime checks

The repository targets Python 3.12+. Stop early if the runtime is older.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

python_version = tuple(int(part) for part in sys.version.split()[0].split(".")[:2])
print("python=", sys.version)
if python_version < (3, 12):
    raise RuntimeError(
        "This project requires Python 3.12+. Use a Colab runtime with Python 3.12 or run on Kaggle."
    )

In [ ]:
import torch

print("cuda_available=", torch.cuda.is_available())
print("device=", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("torch=", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Colab before training.")

## Clone or refresh the repository

This is idempotent. If the repo already exists, it pulls the latest `main`.

In [ ]:
REPO_URL = "https://github.com/Nhkp/medical-triage-agent.git"
REPO_DIR = Path("/content/medical-triage-agent")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git checkout; remove it or choose another path"
        )
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(Path.cwd())

## Install only missing training dependencies

Do not reinstall Torch. Colab already provides the CUDA-enabled build, and replacing it can waste time or break the runtime.

In [ ]:
!python -m pip install -q -U \
  datasets peft trl transformers accelerate bitsandbytes pyyaml trackio wrapt huggingface_hub

## Load Hugging Face token

Preferred: put `HF_TOKEN` in a local `.env` file at the repository root. Fallbacks: Colab Secrets, then secure prompt. The token needs read access to the private dataset.

In [ ]:
from getpass import getpass


def load_dotenv(path: Path) -> dict[str, str]:
    values = {}
    if not path.exists():
        return values
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


dotenv = load_dotenv(Path(".env"))
token = dotenv.get("HF_TOKEN")

if not token:
    try:
        from google.colab import userdata
        from google.colab.userdata import SecretNotFoundError, TimeoutException

        token = userdata.get("HF_TOKEN")
    except (ImportError, KeyError, SecretNotFoundError, TimeoutException):
        token = None

if not token:
    token = getpass("HF_TOKEN: ")

os.environ["HF_TOKEN"] = token
os.environ["HF_DATASET_REPO"] = dotenv.get("HF_DATASET_REPO", "Lokhidor/medical-triage-dataset")
os.environ.setdefault(
    "HF_SFT_MODEL_REPO", dotenv.get("HF_SFT_MODEL_REPO", "Lokhidor/medical-triage-qwen3-sft-lora")
)
os.environ.setdefault(
    "HF_DPO_MODEL_REPO", dotenv.get("HF_DPO_MODEL_REPO", "Lokhidor/medical-triage-qwen3-dpo-lora")
)
print("HF token loaded:", bool(os.environ.get("HF_TOKEN")))

## Download the private dataset

The training configs expect local files in `data/processed/training`.

In [ ]:
from huggingface_hub import snapshot_download

DATA_DIR = Path("data/processed/training")
DATA_DIR.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=os.environ["HF_DATASET_REPO"],
    repo_type="dataset",
    local_dir=DATA_DIR,
    token=os.environ["HF_TOKEN"],
    allow_patterns=["sft_*.jsonl", "dpo_*.jsonl", "manifest.json", "README.md"],
)
print("downloaded files:")
for path in sorted(DATA_DIR.glob("*")):
    print("-", path)

## Audit downloaded data

This validates schema, provenance, duplicates, split isolation, and obvious PII findings before training.

In [ ]:
!PYTHONPATH=src python -m medical_triage_agent audit-training-data data/processed/training
!PYTHONPATH=src python -m medical_triage_agent summarize-training-data data/processed/training

## SFT full run on T4

Run the full SFT pass after dataset audit succeeds. The adapter is pushed to `HF_SFT_MODEL_REPO`.

In [ ]:
!python scripts/train_sft.py \
  --config configs/sft_kaggle.yaml \
  --push-to-hub \
  --hub-model-id "$HF_SFT_MODEL_REPO"

## SFT smoke run

Keep this commented smoke command for quick runtime checks before a full run.

In [ ]:
# !python scripts/train_sft.py \
#   --config configs/sft_kaggle.yaml \
#   --max-steps 5 \
#   --max-train-samples 32

## DPO full run after SFT

Run DPO only after `outputs/sft` exists. The aligned adapter is pushed to `HF_DPO_MODEL_REPO`.

In [ ]:
!python scripts/train_dpo.py \
  --config configs/dpo_kaggle.yaml \
  --push-to-hub \
  --hub-model-id "$HF_DPO_MODEL_REPO"

## DPO smoke run

Keep this commented smoke command for quick DPO startup checks.

In [ ]:
# !python scripts/train_dpo.py \
#   --config configs/dpo_kaggle.yaml \
#   --max-steps 5 \
#   --max-train-samples 32

## Optional: deterministic evaluation

Use after the SFT adapter exists.

In [ ]:
!python scripts/evaluate.py \
  --config configs/sft_kaggle.yaml \
  --model sft \
  --adapter-path outputs/sft \
  --output outputs/evaluations/sft.json